In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/sfstata/midwest-quiz-dataset/midwest quiz dataset.csv


In [2]:
import pandas as pd
import numpy as np
import math

midwest = pd.read_csv("/kaggle/input/datasets/sfstata/midwest-quiz-dataset/midwest quiz dataset.csv")

print(midwest.head())

      name  age  apple_pie  potato_salad  sushi  midwest
0     Jeff   32          0             1      1        1
1     Pete   25          1             1      0        1
2     Anne   33          1             1      0        1
3  Natalie   26          0             0      1        0
4   Stella   30          1             1      1        1


In [3]:
def calc_entropy(column):
    """
    Calculate entropy of a binary target column.
    """

    counts = np.bincount(column)
    probabilities = counts / len(column)

    entropy = 0

    for prob in probabilities:
        if prob > 0:
            entropy += prob * math.log(prob, 2)

    return -entropy

In [4]:
def calc_information_gain(data, split_name, target_name):
    """
    Calculate Information Gain.
    """

    original_entropy = calc_entropy(data[target_name])

    values = data[split_name].unique()

    information = 0

    for value in values:
        subset = data[data[split_name] == value]
        probability = len(subset) / len(data)

        information += probability * calc_entropy(subset[target_name])

    return original_entropy - information

In [5]:
def highest_info_gain(data, columns, target_name):

    information_gains = {}

    for col in columns:
        information_gains[col] = calc_information_gain(
            data,
            col,
            target_name
        )

    best_attribute = max(information_gains,
                         key=information_gains.get)

    return best_attribute

In [6]:
def id3(data, original_data, features, target_attribute):

    # -----------------------------------
    # Base Case 1
    # All examples belong to one class
    # -----------------------------------
    if len(data[target_attribute].unique()) == 1:
        return data[target_attribute].iloc[0]

    # -----------------------------------
    # Base Case 2
    # Dataset becomes empty
    # -----------------------------------
    elif len(data) == 0:

        majority_class = original_data[target_attribute].mode()[0]
        return majority_class

    # -----------------------------------
    # Base Case 3
    # No more attributes
    # -----------------------------------
    elif len(features) == 0:

        majority_class = data[target_attribute].mode()[0]
        return majority_class

    # -----------------------------------
    # Recursive Step
    # -----------------------------------
    else:

        best_feature = highest_info_gain(
            data,
            features,
            target_attribute
        )

        tree = {best_feature: {}}

        remaining_features = [f for f in features if f != best_feature]

        for value in data[best_feature].unique():

            subset = data[data[best_feature] == value]

            subtree = id3(
                subset,
                original_data,
                remaining_features,
                target_attribute
            )

            tree[best_feature][value] = subtree

        return tree

In [7]:
features = [
    'apple_pie',
    'potato_salad',
    'sushi'
]

decision_tree = id3(
    midwest,
    midwest,
    features,
    'midwest'
)

decision_tree

{'sushi': {np.int64(1): {'potato_salad': {np.int64(1): {'apple_pie': {np.int64(0): np.int64(1),
      np.int64(1): np.int64(0)}},
    np.int64(0): np.int64(0)}},
  np.int64(0): {'potato_salad': {np.int64(1): {'apple_pie': {np.int64(1): np.int64(1),
      np.int64(0): np.int64(1)}},
    np.int64(0): np.int64(1)}}}}

In [8]:
# Pretty Print the tree
from pprint import pprint

pprint(decision_tree)

{'sushi': {np.int64(0): {'potato_salad': {np.int64(0): np.int64(1),
                                          np.int64(1): {'apple_pie': {np.int64(0): np.int64(1),
                                                                      np.int64(1): np.int64(1)}}}},
           np.int64(1): {'potato_salad': {np.int64(0): np.int64(0),
                                          np.int64(1): {'apple_pie': {np.int64(0): np.int64(1),
                                                                      np.int64(1): np.int64(0)}}}}}}


In [9]:
# Predicting using the tree
def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))

    value = sample[attribute]

    subtree = tree[attribute][value]

    return predict(subtree, sample)

In [10]:
sample = {
    'apple_pie':0,
    'potato_salad':0,
    'sushi':1
}

prediction = predict(decision_tree, sample)

print(prediction)

0
